In [1]:
import pyarrow as pa
import pyarrow.ipc as ipc
import pyarrow.compute as pc
import glob
import os
from collections import defaultdict

DATASET_DIR = r"C:\Users\user\Desktop\muvo_huggingface_dataset"

# ─────────────────────────────────────────
# 1. 데이터셋 구조 분석
# ─────────────────────────────────────────

def load_arrow(path: str) -> pa.Table:
    with open(path, "rb") as f:
        reader = ipc.open_stream(f)
        return pa.Table.from_batches(list(reader))


def analyze_dataset(split: str = "train") -> dict:
    """
    split: 'train' or 'test'
    각 run_id별 파일 목록, frame 범위, timestamp 범위, 시계열 이상 여부를 반환.
    """
    files = sorted(glob.glob(os.path.join(DATASET_DIR, f"*{split}*.arrow")))
    print(f"[{split}] 총 파일 수: {len(files)}")

    # run_id → [(file, min_frame, max_frame, min_ts, max_ts, rows)] 매핑
    run_info = defaultdict(list)
    multi_run_files = []          # 한 파일에 여러 run이 섞인 경우
    within_run_gaps = []          # run 내부에서 frame 불연속 구간

    for path in files:
        tbl = load_arrow(path)
        fname = os.path.basename(path)

        run_ids   = tbl.column("run_id").to_pylist()
        frames    = tbl.column("frame").to_pylist()
        timestamps = tbl.column("timestamp").to_pylist()
        unique_runs = sorted(set(run_ids))

        if len(unique_runs) > 1:
            multi_run_files.append((fname, unique_runs))

        for run in unique_runs:
            mask = [r == run for r in run_ids]
            r_frames = [frames[i] for i, m in enumerate(mask) if m]
            r_ts     = [timestamps[i] for i, m in enumerate(mask) if m]
            run_info[run].append({
                "file": fname,
                "min_frame": min(r_frames),
                "max_frame": max(r_frames),
                "min_ts": min(r_ts),
                "max_ts": max(r_ts),
                "rows": len(r_frames),
            })

            # frame 간격 이상 탐지 (정상 간격 = 5)
            for i in range(len(r_frames) - 1):
                gap = r_frames[i + 1] - r_frames[i]
                if gap != 5:
                    within_run_gaps.append({
                        "run": run, "file": fname,
                        "frame_a": r_frames[i], "frame_b": r_frames[i + 1],
                        "gap": gap,
                    })

    # run 전체에 걸친 파일 간 frame 연속성 검사
    between_file_gaps = []
    for run, chunks in run_info.items():
        chunks_sorted = sorted(chunks, key=lambda x: x["min_frame"])
        for i in range(len(chunks_sorted) - 1):
            expected = chunks_sorted[i]["max_frame"] + 5
            actual   = chunks_sorted[i + 1]["min_frame"]
            if actual != expected:
                between_file_gaps.append({
                    "run": run,
                    "file_a": chunks_sorted[i]["file"],
                    "file_b": chunks_sorted[i + 1]["file"],
                    "expected_next_frame": expected,
                    "actual_next_frame": actual,
                    "gap": actual - expected,
                })

    result = {
        "run_info": dict(run_info),
        "multi_run_files": multi_run_files,
        "within_run_gaps": within_run_gaps,
        "between_file_gaps": between_file_gaps,
    }

    # 요약 출력
    print(f"\n  run 수: {len(run_info)}")
    print(f"  한 파일에 여러 run이 섞인 파일: {len(multi_run_files)}")
    for fname, runs in multi_run_files[:5]:
        print(f"    {fname}: {runs}")

    print(f"\n  run 내부 frame 불연속 구간 수: {len(within_run_gaps)}")
    for g in within_run_gaps[:5]:
        print(f"    run={g['run']} file={g['file']} frame {g['frame_a']}→{g['frame_b']} (gap={g['gap']})")

    print(f"\n  파일 간 frame 불연속 구간 수: {len(between_file_gaps)}")
    for g in between_file_gaps[:5]:
        print(f"    run={g['run']} {g['file_a']}→{g['file_b']} expected={g['expected_next_frame']} actual={g['actual_next_frame']}")

    return result


# ─────────────────────────────────────────
# 2. 시계열 정렬: run_id → (frame, timestamp) 순 정렬
# ─────────────────────────────────────────

def build_timeseries(split: str = "train", output_dir: str | None = None) -> dict[str, pa.Table]:
    """
    모든 파일을 읽어 run_id별로 (frame, timestamp) 기준 오름차순 정렬한
    pa.Table을 반환. output_dir 지정 시 run별로 arrow IPC stream 파일로 저장.
    """
    files = sorted(glob.glob(os.path.join(DATASET_DIR, f"*{split}*.arrow")))
    print(f"[{split}] 파일 로드 중 ... ({len(files)}개)")

    # run_id별 행 수집
    run_batches: dict[str, list[dict]] = defaultdict(list)

    for path in files:
        tbl = load_arrow(path)
        run_ids = tbl.column("run_id").to_pylist()
        for i, run in enumerate(run_ids):
            run_batches[run].append(i)
        # 인덱스 대신 실제 테이블 슬라이스를 run별로 분리
        for run in set(run_ids):
            pass  # placeholder; 아래에서 한 번에 처리

    # 실제로는 테이블 전체를 모아 run별 분리 후 정렬
    all_tables: dict[str, list[pa.Table]] = defaultdict(list)
    for path in files:
        tbl = load_arrow(path)
        run_ids_arr = tbl.column("run_id")
        for run in pc.unique(run_ids_arr).to_pylist():
            mask = pc.equal(run_ids_arr, run)
            all_tables[run].append(tbl.filter(mask))

    sorted_tables: dict[str, pa.Table] = {}
    for run, tables in all_tables.items():
        combined = pa.concat_tables(tables)
        # frame 기준 정렬 (같은 frame이면 timestamp 기준)
        sort_idx = pc.sort_indices(combined, sort_keys=[("frame", "ascending"), ("timestamp", "ascending")])
        sorted_tables[run] = combined.take(sort_idx)
        print(f"  run={run}: {sorted_tables[run].num_rows}행, frame=[{sorted_tables[run].column('frame')[0].as_py()},{sorted_tables[run].column('frame')[-1].as_py()}]")

    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
        for run, tbl in sorted_tables.items():
            out_path = os.path.join(output_dir, f"{split}_{run}_timeseries.arrow")
            with pa.OSFile(out_path, "wb") as sink:
                writer = ipc.new_stream(sink, tbl.schema)
                # 청크 단위로 기록 (메모리 효율)
                for batch in tbl.to_batches(max_chunksize=200):
                    writer.write_batch(batch)
                writer.close()
            print(f"  저장 완료: {out_path}")

    return sorted_tables


# ─────────────────────────────────────────
# 3. 시계열 유효성 최종 검증
# ─────────────────────────────────────────

def verify_timeseries(sorted_tables: dict[str, pa.Table]) -> bool:
    """정렬된 테이블이 실제로 단조 증가(monotonic) frame을 가지는지 확인."""
    all_ok = True
    for run, tbl in sorted_tables.items():
        frames = tbl.column("frame").to_pylist()
        violations = [
            (frames[i], frames[i + 1])
            for i in range(len(frames) - 1)
            if frames[i + 1] < frames[i]   # 역전 허용 안 함
        ]
        if violations:
            print(f"[FAIL] run={run}: frame 역전 {len(violations)}곳: {violations[:3]}")
            all_ok = False
        else:
            # 간격 분포 확인
            gaps = set(frames[i + 1] - frames[i] for i in range(len(frames) - 1))
            print(f"[OK]   run={run}: {len(frames)}행, frame 간격 종류={sorted(gaps)}")
    return all_ok


# ─────────────────────────────────────────
# 실행
# ─────────────────────────────────────────

if __name__ == "__main__":
    for split in ("train", "test"):
        print(f"\n{'='*60}")
        print(f"  SPLIT: {split}")
        print(f"{'='*60}")

        print("\n[1] 데이터셋 구조 분석")
        info = analyze_dataset(split)

        print("\n[2] 시계열 정렬")
        sorted_tables = build_timeseries(
            split=split,
            output_dir=os.path.join(DATASET_DIR, "timeseries_output"),
        )

        print("\n[3] 최종 검증")
        ok = verify_timeseries(sorted_tables)
        print("결과:", "모든 run 정상" if ok else "일부 run에 문제 있음")



  SPLIT: train

[1] 데이터셋 구조 분석
[train] 총 파일 수: 455

  run 수: 24
  한 파일에 여러 run이 섞인 파일: 23
    carla-autopilot-multimodal-dataset-train-00016-of-00455.arrow: ['run_015', 'run_020']
    carla-autopilot-multimodal-dataset-train-00041-of-00455.arrow: ['run_011', 'run_015']
    carla-autopilot-multimodal-dataset-train-00056-of-00455.arrow: ['run_011', 'run_027']
    carla-autopilot-multimodal-dataset-train-00074-of-00455.arrow: ['run_023', 'run_027']
    carla-autopilot-multimodal-dataset-train-00097-of-00455.arrow: ['run_007', 'run_023']

  run 내부 frame 불연속 구간 수: 6953
    run=run_015 file=carla-autopilot-multimodal-dataset-train-00016-of-00455.arrow frame 0→3 (gap=3)
    run=run_015 file=carla-autopilot-multimodal-dataset-train-00016-of-00455.arrow frame 3→6 (gap=3)
    run=run_015 file=carla-autopilot-multimodal-dataset-train-00016-of-00455.arrow frame 6→9 (gap=3)
    run=run_015 file=carla-autopilot-multimodal-dataset-train-00016-of-00455.arrow frame 9→12 (gap=3)
    run=run_015 file=ca

KeyboardInterrupt: 

In [ ]:
import os
import glob
import numpy as np
import pyarrow as pa
import pyarrow.ipc as ipc
import h5py
from PIL import Image
import io

# ─────────────────────────────────────────
# 설정
# ─────────────────────────────────────────
DATASET_DIR  = r"C:\Users\user\Desktop\muvo_huggingface_dataset"
OUTPUT_PATH  = os.path.join(DATASET_DIR, "muvo_dataset.h5")

EXCLUDE_RUNS = {"run_015", "run_016"}
KEEP_COLS    = ["run_id", "frame", "timestamp", "image_front", "lidar", "steer", "throttle", "speed_kmh"]
IMG_RESIZE   = (224, 224)

LIDAR_H, LIDAR_W = 64, 512
FOV_UP, FOV_DOWN = 2.0, -24.8


# ─────────────────────────────────────────
# 전처리 함수
# ─────────────────────────────────────────
def decode_image(raw) -> np.ndarray:
    if isinstance(raw, dict):
        raw = raw.get("bytes") or raw.get("path")
    img = Image.open(io.BytesIO(raw)).convert("RGB")
    img = img.resize(IMG_RESIZE, Image.BILINEAR)
    return np.array(img, dtype=np.uint8)


def pointcloud_to_rangeview(points: np.ndarray) -> np.ndarray:
    """(N, 4) → (64, 512) range image"""
    x, y, z = points[:, 0], points[:, 1], points[:, 2]
    r = np.sqrt(x**2 + y**2 + z**2)
    r = np.where(r == 0, 1e-6, r)

    fov_up_rad   = np.radians(FOV_UP)
    fov_down_rad = np.radians(FOV_DOWN)
    fov_range    = fov_up_rad - fov_down_rad

    pitch   = np.arcsin(np.clip(z / r, -1, 1))
    yaw     = np.arctan2(y, x)

    row_idx = ((1.0 - (pitch - fov_down_rad) / fov_range) * (LIDAR_H - 1)).astype(np.int32).clip(0, LIDAR_H - 1)
    col_idx = ((0.5 * (yaw / np.pi + 1.0)) * (LIDAR_W - 1)).astype(np.int32).clip(0, LIDAR_W - 1)

    range_img = np.zeros((LIDAR_H, LIDAR_W), dtype=np.float32)
    range_img[row_idx, col_idx] = r.astype(np.float32)
    return range_img


# ─────────────────────────────────────────
# HDF5 append
# ─────────────────────────────────────────
def append_to_hdf5(h5: h5py.File, key: str, rows: list[dict]):
    """
    key: "{split}/{run_id}" 형태
    HDF5 구조:
      /{split}/{run_id}/image      (N, 224, 224, 3)  uint8
      /{split}/{run_id}/lidar      (N, 64, 512)       float32
      /{split}/{run_id}/steer      (N,)               float32
      /{split}/{run_id}/throttle   (N,)               float32
      /{split}/{run_id}/speed_kmh  (N,)               float32
      /{split}/{run_id}/frame      (N,)               int32
      /{split}/{run_id}/timestamp  (N,)               float64
    """
    images     = np.stack([decode_image(r["image_front"]) for r in rows])
    lidars     = np.stack([
        pointcloud_to_rangeview(np.array(r["lidar"], dtype=np.float32).reshape(-1, 4))
        for r in rows
    ])
    steers     = np.array([r["steer"]     for r in rows], dtype=np.float32)
    throttles  = np.array([r["throttle"]  for r in rows], dtype=np.float32)
    speeds     = np.array([r["speed_kmh"] for r in rows], dtype=np.float32)
    frames     = np.array([r["frame"]     for r in rows], dtype=np.int32)
    timestamps = np.array([r["timestamp"] for r in rows], dtype=np.float64)

    grp = h5.require_group(key)

    def _append(name, data):
        if name not in grp:
            grp.create_dataset(
                name, data=data,
                maxshape=(None,) + data.shape[1:],
                chunks=True, compression="gzip", compression_opts=4
            )
        else:
            ds  = grp[name]
            old = ds.shape[0]
            ds.resize(old + len(data), axis=0)
            ds[old:] = data

    _append("image",     images)
    _append("lidar",     lidars)
    _append("steer",     steers)
    _append("throttle",  throttles)
    _append("speed_kmh", speeds)
    _append("frame",     frames)
    _append("timestamp", timestamps)


# ─────────────────────────────────────────
# 후처리: frame 기준 재정렬 + 중복 제거
# ─────────────────────────────────────────
def postprocess_hdf5(h5: h5py.File, split: str):
    print(f"\n[{split}] 후처리: 정렬 및 중복 제거 중...")

    if split not in h5:
        print(f"  [{split}] 그룹 없음, 스킵")
        return

    for run_id in sorted(h5[split].keys()):
        grp       = h5[f"{split}/{run_id}"]
        frames    = grp["frame"][:]
        sort_idx  = np.argsort(frames, kind="stable")
        _, unique_pos = np.unique(frames[sort_idx], return_index=True)
        final_idx = sort_idx[unique_pos]

        before = len(frames)
        for name in list(grp.keys()):
            data    = grp[name][:]
            cleaned = data[final_idx]
            del grp[name]
            grp.create_dataset(
                name, data=cleaned,
                maxshape=(None,) + cleaned.shape[1:],
                chunks=True, compression="gzip", compression_opts=4
            )

        after = len(final_idx)
        print(f"  {split}/{run_id}: {before}행 → {after}행 (중복 {before - after}개 제거)")


# ─────────────────────────────────────────
# 메인
# ─────────────────────────────────────────
def build_hdf5(split: str):
    files = sorted(glob.glob(os.path.join(DATASET_DIR, f"*{split}*.arrow")))
    print(f"\n[{split}] 총 {len(files)}개 파일 처리 시작")
    print(f"출력: {OUTPUT_PATH}")

    with h5py.File(OUTPUT_PATH, "a") as h5:

        # ── 1. 파일 순회하며 run별 append ──
        for i, path in enumerate(files):
            fname = os.path.basename(path)
            try:
                with open(path, "rb") as f:
                    table = pa.Table.from_batches(list(ipc.open_stream(f)))
            except Exception as e:
                print(f"  [ERROR] {fname}: {e}")
                continue

            run_ids = table.column("run_id").to_pylist()

            for run in sorted(set(run_ids)):
                if run in EXCLUDE_RUNS:
                    continue

                indices = [j for j, r in enumerate(run_ids) if r == run]
                rows = [
                    {col: table.column(col)[j].as_py()
                     for col in KEEP_COLS if col in table.column_names}
                    for j in indices
                ]
                rows.sort(key=lambda x: x["frame"])
                append_to_hdf5(h5, f"{split}/{run}", rows)

            # if (i + 1) % 50 == 0 or (i + 1) == len(files):
            print(f"  [{i+1:>3}/{len(files)}] {fname}")

        # ── 2. 후처리 ──
        postprocess_hdf5(h5, split)

    print(f"\n✅ [{split}] 완료")


if __name__ == "__main__":
    build_hdf5("train")
    build_hdf5("test")

In [1]:
import os
import glob
import pyarrow as pa
import pyarrow.ipc as ipc
import pyarrow.compute as pc

DATASET_DIR  = r"C:\Users\user\Desktop\muvo_huggingface_dataset"
OUTPUT_DIR   = os.path.join(DATASET_DIR, "processed")
os.makedirs(OUTPUT_DIR, exist_ok=True)

EXCLUDE_RUNS = {"run_015", "run_016"}
KEEP_COLS    = ["run_id", "frame", "timestamp", "image_front", "lidar", "steer", "throttle", "speed_kmh"]


def build_arrow(split: str):
    files = sorted(glob.glob(os.path.join(DATASET_DIR, f"*{split}*.arrow")))
    print(f"\n[{split}] 총 {len(files)}개 파일 처리 시작")

    schema = None
    file_handles = {}
    writers = {}

    for i, path in enumerate(files):
        fname = os.path.basename(path)
        try:
            with open(path, "rb") as f:
                table = pa.Table.from_batches(list(ipc.open_stream(f)))
        except Exception as e:
            print(f"  [ERROR] {fname}: {e}")
            continue

        table = table.select([c for c in KEEP_COLS if c in table.column_names])
        if schema is None:
            schema = table.schema

        run_ids = table.column("run_id").to_pylist()
        for run in sorted(set(run_ids)):
            if run in EXCLUDE_RUNS:
                continue

            if run not in writers:
                out_path = os.path.join(OUTPUT_DIR, f"{split}_{run}.arrow")
                fh = open(out_path, "wb")
                file_handles[run] = fh
                writers[run] = ipc.new_stream(fh, schema)

            mask = pc.equal(table.column("run_id"), run)
            run_table = table.filter(mask)
            for batch in run_table.to_batches(max_chunksize=200):
                writers[run].write_batch(batch)

        print(f"  [{i+1:>3}/{len(files)}] {fname}")

    for run, writer in writers.items():
        writer.close()
    for fh in file_handles.values():
        fh.close()
    print(f"\n✅ [{split}] 완료")


if __name__ == "__main__":
    build_arrow("train")
    build_arrow("validation")
    build_arrow("test")


[train] 총 455개 파일 처리 시작
  [  1/455] carla-autopilot-multimodal-dataset-train-00000-of-00455.arrow
  [  2/455] carla-autopilot-multimodal-dataset-train-00001-of-00455.arrow
  [  3/455] carla-autopilot-multimodal-dataset-train-00002-of-00455.arrow
  [  4/455] carla-autopilot-multimodal-dataset-train-00003-of-00455.arrow
  [  5/455] carla-autopilot-multimodal-dataset-train-00004-of-00455.arrow
  [  6/455] carla-autopilot-multimodal-dataset-train-00005-of-00455.arrow
  [  7/455] carla-autopilot-multimodal-dataset-train-00006-of-00455.arrow
  [  8/455] carla-autopilot-multimodal-dataset-train-00007-of-00455.arrow
  [  9/455] carla-autopilot-multimodal-dataset-train-00008-of-00455.arrow
  [ 10/455] carla-autopilot-multimodal-dataset-train-00009-of-00455.arrow
  [ 11/455] carla-autopilot-multimodal-dataset-train-00010-of-00455.arrow
  [ 12/455] carla-autopilot-multimodal-dataset-train-00011-of-00455.arrow
  [ 13/455] carla-autopilot-multimodal-dataset-train-00012-of-00455.arrow
  [ 14/455] c

In [3]:
import pyarrow as pa
import pyarrow.ipc as ipc
import glob
import os
from collections import defaultdict

DATASET_DIR = r"C:\Users\user\Desktop\muvo_huggingface_dataset"
EXCLUDE_RUNS = {"run_015", "run_016"}

for split in ("train", "validation", "test"):
    files = sorted(glob.glob(os.path.join(DATASET_DIR, f"*{split}*.arrow")))
    
    # run별로 (파일번호, min_frame, max_frame) 수집
    run_chunks = defaultdict(list)
    for i, path in enumerate(files):
        with open(path, "rb") as f:
            table = pa.Table.from_batches(list(ipc.open_stream(f)))
        run_ids = table.column("run_id").to_pylist()
        frames  = table.column("frame").to_pylist()
        for run in set(run_ids):
            if run in EXCLUDE_RUNS:
                continue
            r_frames = [frames[j] for j, r in enumerate(run_ids) if r == run]
            run_chunks[run].append((i, min(r_frames), max(r_frames)))

    print(f"\n[{split}]")
    all_ok = True
    for run, chunks in sorted(run_chunks.items()):
        chunks_sorted_by_file = sorted(chunks, key=lambda x: x[0])
        chunks_sorted_by_frame = sorted(chunks, key=lambda x: x[1])
        if chunks_sorted_by_file != chunks_sorted_by_frame:
            print(f"  {run}: ❌ 파일 순서 ≠ frame 순서")
            all_ok = False
        else:
            print(f"  {run}: ✅")
    if all_ok:
        print(f"  → 전체 정상")


[train]
  run_002: ✅
  run_003: ✅
  run_005: ✅
  run_006: ✅
  run_007: ✅
  run_010: ✅
  run_011: ✅
  run_012: ✅
  run_013: ✅
  run_014: ✅
  run_017: ✅
  run_018: ✅
  run_019: ✅
  run_020: ✅
  run_022: ✅
  run_023: ✅
  run_025: ✅
  run_026: ✅
  run_027: ✅
  run_028: ✅
  run_029: ✅
  run_030: ✅
  → 전체 정상

[validation]
  run_008: ✅
  run_009: ✅
  run_024: ✅
  → 전체 정상

[test]
  run_001: ✅
  run_004: ✅
  run_021: ✅
  → 전체 정상


In [2]:
import os
import glob
import pyarrow as pa
import pyarrow.ipc as ipc
import pyarrow.compute as pc

DATASET_DIR = r"C:\Users\user\Desktop\muvo_huggingface_dataset"
OUTPUT_DIR  = os.path.join(DATASET_DIR, "processed")
os.makedirs(OUTPUT_DIR, exist_ok=True)

EXCLUDE_RUNS = {"run_015", "run_016"}
KEEP_COLS    = ["run_id", "frame", "timestamp", "image_front", "lidar", "steer", "throttle", "speed_kmh"]


def build_arrow(split: str):
    files = sorted(glob.glob(os.path.join(DATASET_DIR, f"*{split}*.arrow")))
    print(f"\n[{split}] 총 {len(files)}개 파일 처리 시작")

    # run별로 테이블 수집
    run_tables: dict[str, list[pa.Table]] = {}

    for i, path in enumerate(files):
        fname = os.path.basename(path)
        try:
            with open(path, "rb") as f:
                table = pa.Table.from_batches(list(ipc.open_stream(f)))
        except Exception as e:
            print(f"  [ERROR] {fname}: {e}")
            continue

        # 필요한 컬럼만
        table = table.select([c for c in KEEP_COLS if c in table.column_names])

        run_ids = table.column("run_id").to_pylist()
        for run in sorted(set(run_ids)):
            if run in EXCLUDE_RUNS:
                continue
            mask = pc.equal(table.column("run_id"), run)
            run_table = table.filter(mask)
            if run not in run_tables:
                run_tables[run] = []
            run_tables[run].append(run_table)

        print(f"  [{i+1:>3}/{len(files)}] {fname}")

    # run별로 합치고 frame 기준 정렬 후 저장
    print(f"\n[{split}] run별 정렬 및 저장 중...")
    for run, tables in sorted(run_tables.items()):
        combined = pa.concat_tables(tables)
        frames = combined.column("frame").to_pylist()
        sort_idx = sorted(range(len(frames)), key=lambda x: frames[x])
        combined = combined.take(sort_idx)

        out_path = os.path.join(OUTPUT_DIR, f"{split}_{run}.arrow")
        with open(out_path, "wb") as f:
            writer = ipc.new_stream(f, combined.schema)
            for batch in combined.to_batches(max_chunksize=200):
                writer.write_batch(batch)
            writer.close()

        print(f"  {split}/{run}: {combined.num_rows}행 → {out_path}")

    print(f"\n✅ [{split}] 완료")


if __name__ == "__main__":
    build_arrow("train")
    build_arrow("validation")

    build_arrow("test")


[train] 총 455개 파일 처리 시작
  [  1/455] carla-autopilot-multimodal-dataset-train-00000-of-00455.arrow
  [  2/455] carla-autopilot-multimodal-dataset-train-00001-of-00455.arrow
  [  3/455] carla-autopilot-multimodal-dataset-train-00002-of-00455.arrow
  [  4/455] carla-autopilot-multimodal-dataset-train-00003-of-00455.arrow
  [  5/455] carla-autopilot-multimodal-dataset-train-00004-of-00455.arrow
  [  6/455] carla-autopilot-multimodal-dataset-train-00005-of-00455.arrow
  [  7/455] carla-autopilot-multimodal-dataset-train-00006-of-00455.arrow
  [  8/455] carla-autopilot-multimodal-dataset-train-00007-of-00455.arrow
  [  9/455] carla-autopilot-multimodal-dataset-train-00008-of-00455.arrow
  [ 10/455] carla-autopilot-multimodal-dataset-train-00009-of-00455.arrow
  [ 11/455] carla-autopilot-multimodal-dataset-train-00010-of-00455.arrow
  [ 12/455] carla-autopilot-multimodal-dataset-train-00011-of-00455.arrow
  [ 13/455] carla-autopilot-multimodal-dataset-train-00012-of-00455.arrow
  [ 14/455] c

ArrowInvalid: offset overflow while concatenating arrays

In [5]:
import h5py

OUTPUT_PATH = r"C:\Users\user\Desktop\muvo_huggingface_dataset\muvo_dataset.h5"

with h5py.File(OUTPUT_PATH, "r") as h5:
    for split in h5.keys():
        print(f"\n[{split}]")
        for run_id in sorted(h5[split].keys()):
            grp = h5[f"{split}/{run_id}"]
            n = grp["frame"].shape[0]
            f_min = grp["frame"][0]
            f_max = grp["frame"][-1]
            print(f"  {run_id}: {n}행, frame=[{f_min}, {f_max}], "
                  f"image={grp['image'].shape}, lidar={grp['lidar'].shape}")


[test]
  run_001: 2400행, frame=[0, 11995], image=(2400, 224, 224, 3), lidar=(2400, 64, 512)
  run_004: 2400행, frame=[0, 11995], image=(2400, 224, 224, 3), lidar=(2400, 64, 512)
  run_021: 2400행, frame=[0, 11995], image=(2400, 224, 224, 3), lidar=(2400, 64, 512)

[train]
  run_002: 2400행, frame=[0, 11995], image=(2400, 224, 224, 3), lidar=(2400, 64, 512)
  run_003: 2400행, frame=[0, 11995], image=(2400, 224, 224, 3), lidar=(2400, 64, 512)
  run_005: 2400행, frame=[0, 11995], image=(2400, 224, 224, 3), lidar=(2400, 64, 512)
  run_006: 2400행, frame=[0, 11995], image=(2400, 224, 224, 3), lidar=(2400, 64, 512)
  run_007: 2400행, frame=[0, 11995], image=(2400, 224, 224, 3), lidar=(2400, 64, 512)
  run_010: 2400행, frame=[0, 11995], image=(2400, 224, 224, 3), lidar=(2400, 64, 512)
  run_011: 2400행, frame=[0, 11995], image=(2400, 224, 224, 3), lidar=(2400, 64, 512)
  run_012: 3600행, frame=[0, 17995], image=(3600, 224, 224, 3), lidar=(3600, 64, 512)
  run_013: 4800행, frame=[0, 23995], image=(4800,

In [8]:
import h5py
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

OUTPUT_PATH = r"C:\Users\user\Desktop\muvo_dataset_260429\muvo_dataset.h5"

with h5py.File(OUTPUT_PATH, "r") as h5:
    # 구조 트리 출력
    def print_tree(name, obj):
        print(name, "→", obj)
    h5.visititems(print_tree)

test → <HDF5 group "/test" (3 members)>
test/run_001 → <HDF5 group "/test/run_001" (7 members)>
test/run_001/frame → <HDF5 dataset "frame": shape (2400,), type "<i4">
test/run_001/image → <HDF5 dataset "image": shape (2400, 224, 224, 3), type "|u1">
test/run_001/lidar → <HDF5 dataset "lidar": shape (2400, 64, 512), type "<f4">
test/run_001/speed_kmh → <HDF5 dataset "speed_kmh": shape (2400,), type "<f4">
test/run_001/steer → <HDF5 dataset "steer": shape (2400,), type "<f4">
test/run_001/throttle → <HDF5 dataset "throttle": shape (2400,), type "<f4">
test/run_001/timestamp → <HDF5 dataset "timestamp": shape (2400,), type "<f8">
test/run_004 → <HDF5 group "/test/run_004" (7 members)>
test/run_004/frame → <HDF5 dataset "frame": shape (2400,), type "<i4">
test/run_004/image → <HDF5 dataset "image": shape (2400, 224, 224, 3), type "|u1">
test/run_004/lidar → <HDF5 dataset "lidar": shape (2400, 64, 512), type "<f4">
test/run_004/speed_kmh → <HDF5 dataset "speed_kmh": shape (2400,), type "<f4

In [3]:
import glob
import os

DATASET_DIR = r"C:\Users\user\Desktop\muvo_huggingface_dataset"
for split in ("train", "validation", "test"):
    files = glob.glob(os.path.join(DATASET_DIR, f"*{split}*.arrow"))
    print(f"{split}: {len(files)}개")

train: 455개
validation: 70개
test: 49개


In [5]:
import pyarrow as pa
import pyarrow.ipc as ipc
import numpy as np
import glob
import os
from collections import Counter

DATASET_DIR = r"C:\Users\user\Desktop\muvo_huggingface_dataset"

files = sorted(glob.glob(os.path.join(DATASET_DIR, "*train*.arrow")))[:3]

for path in files:
    with open(path, "rb") as f:
        table = pa.Table.from_batches(list(ipc.open_stream(f)))
    
    run_ids = table.column("run_id").to_pylist()
    frames  = table.column("frame").to_pylist()
    
    print(f"\n파일: {os.path.basename(path)}")
    
    for run in sorted(set(run_ids)):
        r_frames = sorted([frames[i] for i, r in enumerate(run_ids) if r == run])
        gaps = [r_frames[i+1] - r_frames[i] for i in range(len(r_frames)-1)]
        gap_counts = Counter(gaps)
        print(f"  run={run}: {len(r_frames)}행, frame 간격 분포={dict(sorted(gap_counts.items()))}")


파일: carla-autopilot-multimodal-dataset-train-00000-of-00455.arrow
  run=run_020: 113행, frame 간격 분포={5: 112}

파일: carla-autopilot-multimodal-dataset-train-00001-of-00455.arrow
  run=run_020: 113행, frame 간격 분포={5: 112}

파일: carla-autopilot-multimodal-dataset-train-00002-of-00455.arrow
  run=run_020: 213행, frame 간격 분포={5: 212}


In [7]:
import pyarrow as pa
import pyarrow.ipc as ipc
import glob
import os
from collections import defaultdict

DATASET_DIR = r"C:\Users\user\Desktop\muvo_huggingface_dataset"
files = sorted(glob.glob(os.path.join(DATASET_DIR, "*train*.arrow")))[:20]

# run별로 파일 걸쳐서 frame 수집
run_frames = defaultdict(list)

for path in files:
    with open(path, "rb") as f:
        table = pa.Table.from_batches(list(ipc.open_stream(f)))
    run_ids = table.column("run_id").to_pylist()
    frames  = table.column("frame").to_pylist()
    for run, frame in zip(run_ids, frames):
        run_frames[run].append(frame)

# 파일 걸친 전체 gap 분석
print("파일 경계 포함 전체 gap 분포:")
for run, flist in sorted(run_frames.items()):
    flist_sorted = sorted(flist)
    gaps = [flist_sorted[i+1] - flist_sorted[i] for i in range(len(flist_sorted)-1)]
    abnormal = [g for g in gaps if g != 5]
    if abnormal:
        from collections import Counter
        print(f"  run={run}: 비정상 gap {len(abnormal)}개 → {dict(Counter(abnormal).most_common(5))}")
    else:
        print(f"  run={run}: 정상 ✅")

파일 경계 포함 전체 gap 분포:
  run=run_015: 비정상 gap 650개 → {3: 650}
  run=run_020: 정상 ✅


In [8]:
import pyarrow as pa
import pyarrow.ipc as ipc
import glob
import os

DATASET_DIR = r"C:\Users\user\Desktop\muvo_huggingface_dataset"

for split in ("train", "test"):
    files = sorted(glob.glob(os.path.join(DATASET_DIR, f"*{split}*.arrow")))
    run_ids = set()
    for path in files:
        with open(path, "rb") as f:
            table = pa.Table.from_batches(list(ipc.open_stream(f)))
        run_ids.update(table.column("run_id").to_pylist())
    print(f"[{split}] run 수: {len(run_ids)}, 목록: {sorted(run_ids)}")

[train] run 수: 24, 목록: ['run_002', 'run_003', 'run_005', 'run_006', 'run_007', 'run_010', 'run_011', 'run_012', 'run_013', 'run_014', 'run_015', 'run_016', 'run_017', 'run_018', 'run_019', 'run_020', 'run_022', 'run_023', 'run_025', 'run_026', 'run_027', 'run_028', 'run_029', 'run_030']
[test] run 수: 3, 목록: ['run_001', 'run_004', 'run_021']


In [9]:
# analyze_dataset 결과 받아서 run별로 집계
from collections import Counter

# within_run_gaps에서 run별 카운트
within_by_run = Counter(g["run"] for g in info["within_run_gaps"])
print("within_run_gaps run별 분포:", dict(within_by_run))

# between_file_gaps에서 run별 카운트
between_by_run = Counter(g["run"] for g in info["between_file_gaps"])
print("between_file_gaps run별 분포:", dict(between_by_run))

within_run_gaps run별 분포: {'run_015': 3974, 'run_016': 2979}
between_file_gaps run별 분포: {'run_015': 25, 'run_016': 20}
